# SHAP for the saved RoBERTa-base (class weights, seed 42)

Explains the model saved in `Exp_1_4_run_tram_2_epoc_30/roberta_pw_s42`. Nothing is trained; the notebook only reads from Drive and writes to `Exp_3_0_shap`.

Run the cells **top to bottom, one at a time**.

- **Which sentences:** the 725 test rows in a random order (seed 42); the first `N_SAMPLES` are explained. The rule is fixed in the code and does not look at correctness or labels. Raising `N_SAMPLES` later keeps the same first rows.
- **What is explained:** the logit (score before sigmoid) of the model's top-1 predicted label, no threshold.
- **Checks saved per sentence:** `additivity_error` (SHAP values + base value vs the model's score for the original text) and `rebuilt_matches_text` (whether SHAP's rebuilt sentence is identical to the original; if false, the explanation is for the rebuilt text).
- Re-running resumes: rows already in the `.jsonl` file are skipped.

In [ ]:
# Run once if the next cell says "No module named 'shap'"
!pip install -q shap

In [ ]:
# Cell 1 - imports and Drive
import os, json, ast
import numpy as np
import pandas as pd
import torch
import shap
from transformers import AutoTokenizer, AutoModelForSequenceClassification

try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    pass


In [ ]:
# Cell 2 - paths, selection rule, load data and model
DATA_DIR  = "/content/drive/MyDrive/KDU/Thesis/Dataset"
MODEL_DIR = "/content/drive/MyDrive/KDU/Thesis/Exp_1_4_run_tram_2_epoc_30/roberta_pw_s42"
OUT_DIR   = "/content/drive/MyDrive/KDU/Thesis/Exp_3_0_shap"
OUT_FILE  = f"{OUT_DIR}/shap_roberta_pw_s42.jsonl"        # one line per explained sentence; re-running resumes
os.makedirs(OUT_DIR, exist_ok=True)

# ---- SELECTION RULE (fixed before looking at any explanation) ----
# Random order of the 725 test rows with seed 42; we explain the first N_SAMPLES rows of that order.
# Not chosen by correctness, label or how nice the explanation looks. Raising N later keeps the same first rows.
SEED, N_SAMPLES = 42, 3          # 3 = quick test. After you see how long it takes, change to e.g. 100 and re-run.
MAX_LENGTH = 128                  # same as training

# ---- load data, labels, model ----
test_df = pd.read_csv(f"{DATA_DIR}/tram_test.tsv", sep="\t").rename(columns={"text1": "text"})
texts   = test_df["text"].tolist()
classes = np.load(f"{MODEL_DIR}/mlb_classes.npy", allow_pickle=True)
Y_test  = np.load(f"{MODEL_DIR}/test_labels.npy")
assert len(texts) == 725 and Y_test.shape == (725, 180) and len(classes) == 180

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tok    = AutoTokenizer.from_pretrained(MODEL_DIR)
model  = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR).float().to(device).eval()
assert model.config.num_labels == 180

n_long = sum(len(tok(t)["input_ids"]) > MAX_LENGTH for t in texts)


In [ ]:
# Cell 3 - explain the selected sentences (SEED / N_SAMPLES are set in Cell 2)
n_nonascii = sum(any(ord(ch) > 127 for ch in t) for t in texts)
print(f"device={device} | test sentences longer than {MAX_LENGTH} tokens (the model cuts these): {n_long} | "
      f"sentences with non-ASCII characters (curly quotes etc.): {n_nonascii}")

def logits_fn(batch_texts):
    """Raw logits (before sigmoid) for a list of strings. SHAP explains the logit of ONE label."""
    batch_texts = [str(t) for t in batch_texts]
    out = []
    with torch.no_grad():
        for s in range(0, len(batch_texts), 64):
            enc = tok(batch_texts[s:s + 64], truncation=True, max_length=MAX_LENGTH,
                      padding=True, return_tensors="pt").to(device)
            out.append(model(**enc).logits.cpu().numpy())
    return np.concatenate(out)

masker = shap.maskers.Text(tok)

# ---- resume support ----
done = set()
if os.path.exists(OUT_FILE):
    done = {json.loads(l)["row"] for l in open(OUT_FILE) if l.strip()}
order = np.random.RandomState(SEED).permutation(len(texts))[:N_SAMPLES]
print(f"explaining {len(order)} rows, {len(done)} already done")

for n, row in enumerate(order):
    row = int(row)
    if row in done:
        continue
    text = texts[row]
    full = logits_fn([text])[0]                       # all 180 logits for this sentence
    k = int(full.argmax())                            # top-1 predicted label (ranking, no threshold)
    prob = float(1 / (1 + np.exp(-full[k])))
    gold = [str(c) for c in classes[Y_test[row] == 1]]

    explainer = shap.Explainer(lambda b, k=k: logits_fn(b)[:, k], masker, algorithm="partition")
    sv = explainer([text])[0]                         # values: one number per token piece, for label k's logit
    add_err = float(abs(sv.values.sum() + sv.base_values - full[k]))   # SHAP values + base should equal the logit

    rec_tokens = [str(t) for t in sv.data]          # SHAP rebuilds the sentence from these pieces
    rec = {"row": row, "text": text, "pred_label": str(classes[k]), "pred_prob": prob, "pred_logit": float(full[k]),
           "gold_labels": gold, "top1_correct": str(classes[k]) in gold,
           "tokens": [str(t) for t in sv.data], "shap_values": [float(v) for v in sv.values],
           "base_value": float(sv.base_values), "additivity_error": add_err,
           "rebuilt_matches_text": "".join(rec_tokens) == text}
    with open(OUT_FILE, "a") as f:
        f.write(json.dumps(rec) + "\n")
    print(f"[{n+1}/{len(order)}] row {row} pred={rec['pred_label']} (p={prob:.3f}) gold={gold} "
          f"correct={rec['top1_correct']} tokens={len(rec['tokens'])} additivity_err={add_err:.1e} "
          f"rebuilt_text_same_as_original={rec['rebuilt_matches_text']}")
    if n < 3:
        try: shap.plots.text(sv)                      # on-screen view of the first few
        except Exception as e: print("  (plot skipped:", type(e).__name__, ")")

recs = [json.loads(l) for l in open(OUT_FILE) if l.strip()]
print(f"\n{len(recs)} sentences in {OUT_FILE}; top-1 correct in {sum(r['top1_correct'] for r in recs)} of them; "
      f"max additivity error {max(r['additivity_error'] for r in recs):.1e}")
bad = [r["row"] for r in recs if not r["rebuilt_matches_text"]]
print("rows where SHAP's rebuilt text differs from the original (explanation is for the rebuilt text):", bad)
